# yt2drive — YouTube playlist → Google Drive

Downloads every track from a playlist as tagged `.m4a` audio with cover art, straight into a Google Drive folder. Re-run it any time: it only fetches what's new.

**Why Colab is a good place to run this:** both halves of the job happen inside Google's network. The download from YouTube and the write to Drive never touch your home connection, so your own bandwidth and battery are irrelevant and the transfer runs at datacenter speed.

**Run the cells in order.** Steps 1–3 are one-time per session; step 4 is the one you re-run.

---
*Download content you have the rights to. Fetching copyrighted material you don't own a licence for is against YouTube's Terms of Service.*

## Step 1 — Install

Takes about 30 seconds. Point `REPO_URL` at your own fork/copy.

In [ ]:
#@title Install yt2drive + ffmpeg { display-mode: "form" }
REPO_URL = "https://github.com/faisal-saddique/yt2drive"  #@param {type:"string"}

import shutil, subprocess, sys

if shutil.which("ffmpeg") is None:
    print("Installing ffmpeg...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)

# --upgrade matters: YouTube changes its player often and an out-of-date
# yt-dlp is the single most common cause of extraction failures.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     f"git+{REPO_URL}.git", "yt-dlp"],
    check=True,
)

import yt_dlp
print(f"✓ ffmpeg  {shutil.which('ffmpeg')}")
print(f"✓ yt-dlp  {yt_dlp.version.__version__}")
print("✓ yt2drive ready")

## Step 2 — Choose where to save

**Google Drive** — the recommended default. A popup asks you to pick your Google
account and approve access (Colab's built-in Drive mount — no GCP project, OAuth
screen, or credentials to set up). Your Drive appears as a normal folder at
`/content/drive/MyDrive`.

**Colab local storage** — no Google account needed. Files stay on the Colab VM's
disk and you download them as a zip in Step 6. This disk is wiped when the
runtime recycles, so grab the zip before you disconnect — it isn't for long-term
storage.


In [ ]:
#@title Where to save { display-mode: "form" }
SAVE_TO = "Google Drive"  #@param ["Google Drive", "Colab local storage"]

if SAVE_TO == "Google Drive":
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Skipping Drive mount \u2014 saving to local Colab storage instead.")
    print("Remember: this disk is wiped on disconnect. Download the zip in Step 6 before you're done.")


## Step 3 — Settings

`FOLDER_NAME` is the subfolder your library is organised under \u2014 inside
*My Drive* if you chose Drive, or inside Colab's local storage otherwise. It's
created if it doesn't exist.


In [ ]:
#@title Playlist and destination { display-mode: "form" }
PLAYLIST_URL = "https://www.youtube.com/playlist?list=PLGlK3JqJXED9OP9KqQTw0qbRz9JF9Wb-C"  #@param {type:"string"}
FOLDER_NAME = "Music/yt2drive"  #@param {type:"string"}
AUDIO_FORMAT = "m4a"  #@param ["m4a", "opus", "mp3"]
PARALLEL_DOWNLOADS = 3  #@param {type:"slider", min:1, max:8, step:1}
DEDUPE_BY_TITLE = False  #@param {type:"boolean"}

from pathlib import Path

if SAVE_TO == "Google Drive":
    DEST = Path("/content/drive/MyDrive") / FOLDER_NAME
else:
    DEST = Path("/content/yt2drive-library") / FOLDER_NAME
DEST.mkdir(parents=True, exist_ok=True)

# Downloads land here first and are moved to the destination only once complete
# and tagged. Writing straight to a Drive mount is slow and leaves half-files
# behind if the session drops.
STAGING = Path("/content/yt2drive-staging")
STAGING.mkdir(parents=True, exist_ok=True)

print(f"Saving to:   {SAVE_TO}")
print(f"Destination: {DEST}")
print(f"Format:      {AUDIO_FORMAT}")
existing = list(DEST.rglob(f"*.{AUDIO_FORMAT}"))
print(f"Already there: {len(existing)} file(s)")


## Step 4 — Sync

Run this cell whenever you want to catch up. Safe to interrupt — progress is saved after every track, and re-running resumes where it stopped.

> If you see **"Sign in to confirm you're not a bot"**, skip to the cookies cell below. It happens because Colab runs on datacenter IPs that YouTube treats with suspicion.

In [ ]:
#@title Run sync
import sys
from yt2drive.cli import main

argv = [
    "sync", PLAYLIST_URL,
    "--dest", str(DEST),
    "--format", AUDIO_FORMAT,
    "--workers", str(PARALLEL_DOWNLOADS),
    "--staging", str(STAGING),
]
if DEDUPE_BY_TITLE:
    argv.append("--dedupe-by-title")
try:
    COOKIES_PATH  # defined by the cookies cell, if you ran it
    argv += ["--cookies", COOKIES_PATH]
except NameError:
    pass

code = main(argv)
print(f"\nexit code: {code}")

## Step 5 — Check the library

In [ ]:
from yt2drive.cli import main
main(["status", "--dest", str(DEST), "--failures"])

## Step 6 — Download your library (local save only)

Only needed if you picked **Colab local storage** in Step 2 \u2014 Drive already
has your files. Zips up `DEST` and downloads it through the browser.


In [ ]:
#@title Zip and download { display-mode: "form" }
if SAVE_TO == "Google Drive":
    print("Saved to Google Drive \u2014 nothing to download.")
else:
    import shutil
    from google.colab import files

    zip_base = "/content/yt2drive-export"
    archive = shutil.make_archive(zip_base, "zip", root_dir=DEST)
    print(f"Zipped {DEST} -> {archive}")
    files.download(archive)


---
## If YouTube asks you to prove you're not a bot

Give it a signed-in session:

1. Install a **"Get cookies.txt LOCALLY"** extension in your own browser.
2. Open `youtube.com` while signed in, and export `cookies.txt`.
3. Run the cell below and upload that file.
4. Re-run **Step 4**.

Use a throwaway Google account if you'd rather not hand a session cookie to a Colab VM. The file lives only in this session's scratch space and disappears when the runtime is recycled — don't commit it to your repo (`.gitignore` already blocks it).

Dropping `PARALLEL_DOWNLOADS` to 1 also helps on its own.

In [ ]:
#@title Upload cookies.txt
from google.colab import files

uploaded = files.upload()
COOKIES_PATH = "/content/" + next(iter(uploaded))
print(f"Using cookies from {COOKIES_PATH} — now re-run Step 4.")

---
## Extras

**Several playlists at once** — pass more URLs; shared tracks are downloaded once:
```python
main(["sync", URL_A, URL_B, "--dest", str(DEST), "--staging", str(STAGING)])
```

**Preview without downloading:**
```python
main(["sync", PLAYLIST_URL, "--dest", str(DEST), "--dry-run"])
```

**You added files to the Drive folder by hand** — teach the manifest about them so they're never re-downloaded:
```python
main(["verify", "--dest", str(DEST)])
```

**Retry everything that failed:**
```python
main(["sync", PLAYLIST_URL, "--dest", str(DEST), "--retry-failed"])
```